In [1]:
import faiss
import json
import requests
from sentence_transformers import SentenceTransformer
from openai import OpenAI

c:\Users\User\anaconda3\envs\dsgp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
ENDPOINT = "https://openrouter.ai/api/v1"
API_KEY = "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"
LLM_MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY,
)

TOP_K = 5

embedding_model = SentenceTransformer(EMBEDDING_MODEL)


In [3]:
# Load FAISS index + chunks

def load_index(index_dir):
    index = faiss.read_index(f"{index_dir}/index.faiss")

    with open(f"{index_dir}/chunks.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return index, metadata["chunks"]

In [4]:
# Retrieve relevant chunks

def retrieve_chunks(query, index, chunks, top_k=3):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    retrieved = [chunks[i] for i in indices[0]]
    return retrieved

In [5]:
# Call Model

def call_llm(context_chunks, user_query):
    context = "\n\n".join(context_chunks)

    prompt = f"""
You are a supportive speech feedback assistant.

Use ONLY the context below to answer.
Only provide solution.
Do NOT add medical claims.
Do NOT diagnose.
Keep the tone encouraging and practical.

Context:
{context}

User request:
{user_query}

Answer:
"""

    print(prompt)
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.4,
    )

    return response.choices[0].message.content



In [6]:
# Full RAG pipeline

def run_rag(index_dir, user_query):
    index, chunks = load_index(index_dir)
    retrieved_chunks = retrieve_chunks(user_query, index, chunks, TOP_K)
    response = call_llm(retrieved_chunks, user_query)

    return response

In [7]:
# RUN

if __name__ == "__main__":

    problem = "The user uses many filler words"
    
    query = (
        problem,
        "Provide simple improvement advice."
    )

    index_path = "RAG_knowledge/index/filler_words"

    answer = run_rag(index_path, query)
    print("\n--- LLM RESPONSE ---\n")
    print(answer)


You are a supportive speech feedback assistant.

Use ONLY the context below to answer.
Only provide solution.
Do NOT add medical claims.
Do NOT diagnose.
Keep the tone encouraging and practical.

Context:
--- problem_type: speaking_habit primary_issue: filler_words related_issues: pacing, nervousness, fluency severity_range: mild_to_high audience: students, professionals, english_learners context: presentations, interviews, meetings, conversations non_diagnostic: true --- # Filler Words in Spoken Communication ## Overview Filler words are short, often unconscious sounds or words that speakers insert into speech while thinking about what to say next. Common examples include “um”, “uh”, “like”, “you know”, “so”, and “actually”. While filler words are a natural part of everyday communication and are used by almost everyone, frequent or excessive use can reduce clarity, weaken perceived confidence, and distract listeners from the main message. This document provides a practical, non-judgm